<a href="https://colab.research.google.com/github/Junhojuno/pytorch-tutorial/blob/master/mnist_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

import torch.nn.init # 이건 뭐지..?

In [0]:
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(777) # random value 고정
if device == "cuda":
  torch.cuda.manual_seed_all(777)

In [0]:
# parameter
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [4]:
# MNIST dataset
# 그냥 가져오면 tensor형태가 아니므로 ToTensor로 transform해주는 것이다.
mnist_train = datasets.MNIST(root='MNIST_data/',
                             train = True,
                             transform=transforms.ToTensor(),
                             download=True)

mnist_test = datasets.MNIST(root='MNIST_data/',
                             train = False,
                             transform=transforms.ToTensor(),
                             download=True)

0it [00:00, ?it/s]

9920512it [00:01, 8826156.35it/s]                            


Extracting MNIST_data/MNIST/raw/train-images-idx3-ubyte.gz


  0%|          | 0/28881 [00:00<?, ?it/s]

32768it [00:00, 134520.14it/s]           
  0%|          | 0/1648877 [00:00<?, ?it/s]

Extracting MNIST_data/MNIST/raw/train-labels-idx1-ubyte.gz


1654784it [00:00, 2226170.37it/s]                            
0it [00:00, ?it/s]

Extracting MNIST_data/MNIST/raw/t10k-images-idx3-ubyte.gz


8192it [00:00, 50481.96it/s]            


Extracting MNIST_data/MNIST/raw/t10k-labels-idx1-ubyte.gz
Processing...
Done!


In [0]:
# data loader
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size = batch_size,
                                          shuffle=True,
                                          drop_last=True)

In [0]:
# CNN 
class CNN(nn.Module):
  
  def __init__(self):
    super(CNN, self).__init__() # 이거 빠뜨리면 학습이 되지 않는다.
    self.layer1 = nn.Sequential(nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
                                nn.ReLU(),
                                nn.MaxPool2d(2)
                               )
    self.layer2 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
                                nn.ReLU(),
                                nn.MaxPool2d(2) # maxpool2d의 stride는 default가 kernel_size와 같다.
                               ) # 여기까지 output shape이 (batch_size, 64, 7, 7)
    self.fc = nn.Linear(64*7*7,10, bias=True)
    torch.nn.init.xavier_uniform_(self.fc.weight)
    
  def forward(self, x):
    out = self.layer1(x) # input x가 layer1을 다 통과하고 나온 값이 out이다라는 의미
    out = self.layer2(out)
    
    out = out.view(out.size(0), -1) # out.size(0)는 batch_size를 의미
    out = self.fc(out)
    return out

In [9]:
# model 생성
model = CNN().to(device)
model

CNN(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Linear(in_features=3136, out_features=10, bias=True)
)

In [0]:
cost_func = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
# traning code
total_batch = len(data_loader)

for epoch in range(training_epochs):
  avg_cost = 0
  
  for X, Y in data_loader:
    X = X.to(device) # torch cuda tensor 연산
    Y = Y.to(device)
    
    optimizer.zero_grad() # 꼭 설정해줘야함!! 안해주면 학습이 안됨.
    hypothesis = model(X)
    
    cost = cost_func(hypothesis, Y)
    cost.backward()
    optimizer.step()
    
    avg_cost += cost / total_batch
    
  print('[Epoch:{}] cost = {}'.format(epoch+1, avg_cost)) # 한 epoch이 끝나면 cost계산

print('Learning Finished')

[Epoch:1] cost = 0.22380822896957397
[Epoch:2] cost = 0.06039419025182724
[Epoch:3] cost = 0.044629134237766266
[Epoch:4] cost = 0.03588121384382248
[Epoch:5] cost = 0.03004571981728077
[Epoch:6] cost = 0.02608097717165947
[Epoch:7] cost = 0.020601388067007065
[Epoch:8] cost = 0.017544269561767578
[Epoch:9] cost = 0.014817601069808006
[Epoch:10] cost = 0.012949734926223755
[Epoch:11] cost = 0.010524006560444832
[Epoch:12] cost = 0.009743190370500088
[Epoch:13] cost = 0.008462120778858662
[Epoch:14] cost = 0.0068532321602106094
[Epoch:15] cost = 0.005049408879131079
Learning Finished


In [12]:
with torch.no_grad():
  X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
  Y_test = mnist_test.test_labels.to(device)
  
  prediction = model(X_test)
  correct_prediction = torch.argmax(prediction, 1) == Y_test
  accuracy = correct_prediction.float().mean()
  print("Accuracy : ", accuracy.item())

/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:58: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:48: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


Accuracy :  0.9884999990463257


### Layer를 좀 더 깊게 쌓아보자

In [0]:
# CNN 
class CNN(nn.Module):
  
  def __init__(self):
    super(CNN, self).__init__() # 이거 빠뜨리면 학습이 되지 않는다.
    self.layer1 = nn.Sequential(nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1),
                                nn.ReLU(),
                                nn.MaxPool2d(2)
                               )
    self.layer2 = nn.Sequential(nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
                                nn.ReLU(),
                                nn.MaxPool2d(2)
                               ) # 여기까지 output shape이 (batch_size, 64, 7, 7)
    self.layer3 = nn.Sequential(nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
                                nn.ReLU(),
                                nn.MaxPool2d(2)
                               )  # 여기까지 output shape이 (batch_size, 128, 3, 3)
    self.fc1 = nn.Linear(128*3*3, 625, bias=True)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(625, 10, bias=True)
    torch.nn.init.xavier_uniform_(self.fc1.weight)
    torch.nn.init.xavier_uniform_(self.fc2.weight)
    
  def forward(self, x):
    out = self.layer1(x) # input x가 layer1을 다 통과하고 나온 값이 out이다라는 의미
    out = self.layer2(out)
    out = self.layer3(out)
    
    out = out.view(out.size(0), -1) # out.size(0)는 batch_size를 의미
    out = self.fc1(out)
    out = self.relu(out)
    out = self.fc2(out)
    return out

In [20]:
# model 생성
model = CNN().to(device)
model

CNN(
  (layer1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer3): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=1152, out_features=625, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=625, out_features=10, bias=True)
)

In [25]:
# model을 만들고 training전에 샘플로 테스트를 해본다
# 잘돌아가나
value = torch.Tensor(1,1,28,28).to(device)
print(model(value).shape)

torch.Size([1, 10])


In [0]:
cost_func = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [22]:
# traning code
total_batch = len(data_loader)

for epoch in range(training_epochs):
  avg_cost = 0
  
  for X, Y in data_loader:
    X = X.to(device) # torch cuda tensor 연산
    Y = Y.to(device)
    
    optimizer.zero_grad() # 꼭 설정해줘야함!! 안해주면 학습이 안됨.
    hypothesis = model(X)
    
    cost = cost_func(hypothesis, Y)
    cost.backward()
    optimizer.step()
    
    avg_cost += cost / total_batch
    
  print('[Epoch:{}] cost = {}'.format(epoch+1, avg_cost)) # 한 epoch이 끝나면 cost계산

print('Learning Finished')

[Epoch:1] cost = 0.16177113354206085
[Epoch:2] cost = 0.04231693968176842
[Epoch:3] cost = 0.028634872287511826
[Epoch:4] cost = 0.02161252312362194
[Epoch:5] cost = 0.018795832991600037
[Epoch:6] cost = 0.014902991242706776
[Epoch:7] cost = 0.011888774111866951
[Epoch:8] cost = 0.010675027966499329
[Epoch:9] cost = 0.010117881000041962
[Epoch:10] cost = 0.008148315362632275
[Epoch:11] cost = 0.008668183349072933
[Epoch:12] cost = 0.006648878566920757
[Epoch:13] cost = 0.005850268993526697
[Epoch:14] cost = 0.004677235148847103
[Epoch:15] cost = 0.005347175989300013
Learning Finished


In [23]:
with torch.no_grad():
  X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
  Y_test = mnist_test.test_labels.to(device)
  
  prediction = model(X_test)
  correct_prediction = torch.argmax(prediction, 1) == Y_test
  accuracy = correct_prediction.float().mean()
  print("Accuracy : ", accuracy.item())

/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:58: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
/usr/local/lib/python3.6/dist-packages/torchvision/datasets/mnist.py:48: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


Accuracy :  0.9907999634742737
